# 02_customer_impact.ipynb
Objectives:
- Validate H2: SLA violations lead to worse customer experience and retention.
- Quantify how customer harm increases with delay severity (dose–response).
- Provide multi-level evidence:
  - Level 1: Descriptive signposts (on-time vs delayed).
  - Level 2: Stratified comparison (approx. matched analysis).
  - Level 3: Within-seller before/after analysis around severe events.
  - Level 4: Delay bucket dose–response.
- Prepare artifacts for downstream ROI and intervention simulations.

Notes:
- This notebook operates at the order-level and links SLA metrics to
  customer experience outcomes (reviews, cancellations, repeat purchases).
- It relies on the seller-level SLA pipeline already validated in
  `01_seller_risk.ipynb`.
- Economic impact / ROI will be handled in later modules.

### 0. Import & Settings

In [2]:
import pandas as pd
import numpy as np
import altair as alt
from IPython.core.interactiveshell import InteractiveShell
from pathlib import Path
import sys

pd.set_option("display.max_columns", 50)
InteractiveShell.ast_node_interactivity = "all"
alt.data_transformers.disable_max_rows()


NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
    

# Import project modules
from config import DATA_INTERIM, DATA_PROCESSED
from features.seller_metrics import validate_orders_sellers
from features.customer_impact import (
    build_order_customer_panel,
#     summarize_cx_by_sla_flag,
#     summarize_cx_by_delay_bucket,
#     summarize_cx_by_strata_and_sla,
#     within_seller_before_after_summary,
)
from data.preprocessing import (
    load_orders_sellers,
)
from data.load_raw import (
    load_reviews,
    load_customers,
)

### 1. Data Load & Panel Construction

This section:
- Loads the base tables required for H2:
  - `orders_sellers`: wide table with SLA metrics at order + seller level.
  - `reviews`: order-level review scores.
  - `customers`: customer dimension (for repeat behavior and geography).
- Validates the SLA-wide table schema using `validate_orders_sellers`.
- Builds an **order-level panel** that links SLA and customer experience:
  - review_score / low-rating flags
  - cancellation flag
  - delay buckets
  - 30-day repeat purchase indicator

In [3]:
# load base tables
orders_sellers = load_orders_sellers()
reviews = load_reviews()
customers = load_customers()

# print basic attributes of the tables
print(
    f"orders_sellers shape: {orders_sellers.shape}, "
    f"reviews shape: {reviews.shape}, "
    f"customers shape: {customers.shape}"
)

# print the time range of the orders_sellers table and display the first few rows
print(
    "orders_sellers time range:",
    orders_sellers["order_purchase_timestamp"].min(),
    "to",
    orders_sellers["order_purchase_timestamp"].max(),
)
orders_sellers.head()

orders_sellers shape: (99441, 11), reviews shape: (99224, 7), customers shape: (99441, 5)
orders_sellers time range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18


,order_id,seller_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delay_days,is_sla_violation,is_severe_violation,has_time_anomaly
0,e481f51cbdc54678b7cc49136f2d6af7,3504c0cb71d7fa48d967e0e4c94d59d9,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,-8.0,False,False,False
1,53cdb2fc8bc7dce0b6741e2150273451,289cdb325fb7e7f891c38608bf9e0962,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,-6.0,False,False,False
2,47770eb9100c2d0c44946d9cf07ec65d,4869f7a5dfa277a7dca6462dcf3b52b2,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,-18.0,False,False,False
3,949d5b44dbf5de918fe9c16f97b45f8a,66922902710d126a0e7d26b0e3805106,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,-13.0,False,False,False
4,ad21c59c0840e6cb83a9ceb5573f8159,2c9e548be18521d1c43cde1c582c6de8,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,-10.0,False,False,False


In [4]:
# validate the orders_sellers table, same as 01
validate_orders_sellers(orders_sellers, gmv_col = None)

orders_sellers contains unexpected order_status values: {'approved'}


In [5]:
# build order-level panel linking SLA and CX
panel = build_order_customer_panel(
    orders_sellers=orders_sellers,
    order_reviews=reviews,
    customers=customers,
    max_horizon_days=30,
    low_rating_threshold=3,
    very_low_rating_threshold=2,
)

panel.head()

,order_id,seller_id,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delay_days,is_sla_violation,is_severe_violation,has_time_anomaly,review_score,review_creation_date,is_low_rating,is_very_low_rating,is_canceled,delay_bucket,customer_city,customer_state,next_order_time,days_to_next_order,repeat_within_horizon
68955,5f79b5b0931d63f1a42989eb65b9da6e,7aa4334be125fcdd2ba64b3180029f14,00012a2ce6f8dcda20d059ce98491703,delivered,2017-11-14 16:08:26,2017-11-28 15:41:30,2017-12-04,-6.0,False,False,False,1.0,2017-11-29,True,True,False,on_time_or_early,osasco,SP,NaT,NaN,False
10070,a44895d095d7e0702b6a162fa2dbeced,2a1348e9addc1af5aaa619b1a3679d6b,000161a058600d5901f007fab4c27140,delivered,2017-07-16 09:40:32,2017-07-25 18:57:33,2017-08-04,-10.0,False,False,False,4.0,2017-07-26,False,False,False,on_time_or_early,itapecerica,MG,NaT,NaN,False
66244,316a104623542e4d75189bb372bc5f8d,46dc3b2cc0980fb8ec44634e21d2718e,0001fd6190edaaf884bcaf3d49edf079,delivered,2017-02-28 11:06:43,2017-03-06 08:57:49,2017-03-22,-16.0,False,False,False,5.0,2017-03-07,False,False,False,on_time_or_early,nova venecia,ES,NaT,NaN,False
43391,5825ce2e88d5346438686b0bba99e5ee,aafe36600ce604f205b86b5084d3d767,0002414f95344307404f0ace7a26f1d5,delivered,2017-08-16 13:09:20,2017-09-13 20:06:02,2017-09-14,-1.0,False,False,False,5.0,2017-09-14,False,False,False,on_time_or_early,mendonca,MG,NaT,NaN,False
5916,0ab7fb08086d4af9141453c91878ed7a,4a3ca9315b744ce9f8e9374361493884,000379cdec625522490c315e70c7a9fb,delivered,2018-04-02 13:42:17,2018-04-13 20:21:08,2018-04-18,-5.0,False,False,False,4.0,2018-04-14,False,False,False,on_time_or_early,sao paulo,SP,NaT,NaN,False


In [6]:
# Basic sanity checks on the panel
panel[["review_score", "is_low_rating", "is_canceled", "repeat_within_horizon"]].isna().mean()

review_score             0.007681
is_low_rating            0.000000
is_canceled              0.000000
repeat_within_horizon    0.000000
dtype: float64

In [8]:
# Delay buckets distribution
panel["delay_bucket"].value_counts(dropna=False).sort_index()

delay_bucket
on_time_or_early    90442
1-2_days_late        1374
3-5_days_late        1403
6+_days_late         3786
NaN                  2987
Name: count, dtype: int64

In [10]:
# Repeat within horizon distribution
panel["repeat_within_horizon"].mean()

np.float64(0.005510440835266821)